# 05 · Computer Vision Architecture Zoo and Detection

Study convolution, pooling, augmentation, seven educational architecture families, a transfer-learning mechanics proxy, one-object detection, IoU, precision/recall, mAP, efficiency, and visual errors.

This is a learning and interview notebook, not a thin launcher. It combines mechanism,
executable evidence, diagnostics, limitations, and experiments. Run from top to bottom;
keep `RUN_TRAINING=False` to inspect committed/CI results, or turn it on to reproduce them.

In [ ]:
# Local/VS Code and Colab bootstrap. No secret or API key is used.
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    repo = Path("/content/ai-rookie-skill-tree-lab")
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/chelsietao/ai-rookie-skill-tree-lab.git", str(repo)],
            check=True,
        )
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[full,dev]"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data/generated/ci"
METRICS_DIR = ROOT / "artifacts/metrics/ci"
RUN_TRAINING = False  # Set True when you want to regenerate the chapter's metrics.
assert (ROOT / "pyproject.toml").exists(), "Open the repository root or notebooks directory."
print({"environment": "Colab" if IN_COLAB else "local/VS Code", "root": str(ROOT)})

## Learning objectives

- Implement and visualize convolution/pooling behavior.
- Locate the defining block in VGG, ResNet, MobileNet, EfficientNet, SqueezeNet, and ViT.
- Compare accuracy, parameters, runtime, and training curves.
- Compute IoU and distinguish mAP@0.50 from mAP@0.50:0.95.
- Inspect detection errors instead of relying on a single score.

In [ ]:
if not (DATA_DIR / "manifest.json").exists():
    subprocess.run(
        [sys.executable, "scripts/prepare_data.py", "--profile", "ci", "--output", str(DATA_DIR)],
        check=True,
    )
subprocess.run([sys.executable, "scripts/validate_data.py", "--data-dir", str(DATA_DIR)], check=True)
manifest = json.loads((DATA_DIR / "manifest.json").read_text())
print(manifest["dataset_version"], manifest["rows"])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

bundle = np.load(DATA_DIR / "product_images.npz")
images, labels = bundle["x"], bundle["y"]
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for axis, image, label in zip(axes.flat, images[:12], labels[:12], strict=True):
    axis.imshow(image.transpose(1, 2, 0)); axis.set_title(f"class {label}"); axis.axis("off")
plt.tight_layout()

## Convolution

A convolution kernel shares weights over spatial locations. Padding controls output size; stride
controls downsampling; channels let filters combine color and learned feature maps.

In [ ]:
sample = torch.tensor(images[0:1].astype("float32") / 255)
edge_kernel = torch.tensor([[-1., -1., -1.], [-1., 8., -1.], [-1., -1., -1.]])
kernel = torch.zeros(1, 3, 3, 3)
kernel[0, 0] = edge_kernel
edges = torch.nn.functional.conv2d(sample, kernel, padding=1)
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(images[0].transpose(1, 2, 0)); axes[0].set_title("input")
axes[1].imshow(edges[0, 0], cmap="coolwarm"); axes[1].set_title("edge response")
plt.show()

## Pooling and augmentation

Max pooling preserves strong local responses; average pooling summarizes a region. Horizontal flips
and Gaussian noise add invariances here. An augmentation is valid only if it preserves the label.

In [ ]:
max_pool = torch.nn.functional.max_pool2d(sample, 2)
avg_pool = torch.nn.functional.avg_pool2d(sample, 2)
print(sample.shape, max_pool.shape, avg_pool.shape)

In [ ]:
metric_path = METRICS_DIR / "vision.json"
if RUN_TRAINING or not metric_path.exists():
    metric_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, "scripts/run_vision.py", "--data-dir", str(DATA_DIR), "--profile", "ci", "--output", str(METRICS_DIR / "vision.json")], check=True)
result = json.loads(metric_path.read_text())
print("config", Path("configs/models/vision.yaml").read_text())
print("loaded", metric_path, "dataset", result.get("_run", {}).get("dataset_version"))

In [ ]:
import pandas as pd
zoo = pd.DataFrame(result["classification"]).T
display(zoo[["accuracy", "parameters", "fit_seconds", "batch32_inference_ms"]].sort_values("accuracy", ascending=False))

## Architecture map

- **VGG:** repeated 3×3 convolution stacks.
- **ResNet:** identity skip connections stabilize deeper optimization.
- **MobileNet:** depthwise spatial convolution plus pointwise channel mixing.
- **EfficientNet-style:** depthwise blocks, smooth activation, squeeze/excitation.
- **SqueezeNet:** fire modules squeeze then expand channels.
- **ViT:** patch embedding, position encoding, multi-head self-attention.

These are intentionally tiny teaching variants, not canonical ImageNet models.

In [ ]:
from rookie_lab.vision.models import model_zoo
for name, model in model_zoo().items():
    print("\n", name, "parameters", sum(p.numel() for p in model.parameters()))
    print(model)

## Quick-profile failures matter

Short training can leave MobileNet/EfficientNet underfit even when simpler models reach perfect
synthetic accuracy. This is evidence about optimization budget, not architecture quality.

In [ ]:
zoo["accuracy"].sort_values().plot.barh(title="Quick-profile architecture accuracy")
plt.xlim(0, 1.05); plt.show()
for name, row in result["classification"].items():
    plt.plot(row["training_loss"], marker="o", label=name)
plt.yscale("log"); plt.legend(ncol=2); plt.title("Training loss"); plt.show()

In [ ]:
display(pd.DataFrame(result["pooling_ablation"]).T[["accuracy", "fit_seconds", "training_loss"]])

## Transfer-learning mechanics proxy

The experiment pretrains a small CNN for two epochs, freezes its convolutional representation,
resets the classifier, and adapts the head for one epoch. Because source and target are the same
synthetic domain, this demonstrates mechanics—not external knowledge transfer.

In [ ]:
transfer = result["transfer_learning_proxy"]
display(pd.DataFrame({
    "frozen_transfer": transfer["frozen_feature_transfer"],
    "scratch": transfer["one_epoch_from_scratch"],
}).T[["accuracy", "fit_seconds", "training_loss"]])
print(transfer["limitation"])

## Intersection over Union

$IoU = area(B_p \cap B_t) / area(B_p \cup B_t)$. A correct class with a poorly localized box can
still fail detection. Higher IoU thresholds demand more precise localization.

In [ ]:
from rookie_lab.vision.experiment import _iou
prediction = torch.tensor([[0.1, 0.1, 0.8, 0.8]])
target = torch.tensor([[0.2, 0.2, 0.9, 0.9]])
print("example IoU", _iou(prediction, target).item())

## Detection metrics

This dataset has exactly one prediction and one target per image. AP is averaged across classes and
IoU thresholds under that simplified protocol. It is not COCO multi-object mAP with NMS/crowds.

In [ ]:
detection = result["object_detection"]
display(pd.Series({key: value for key, value in detection.items() if not isinstance(value, dict) and key != "error_figure"}))
pd.Series(detection["ap_by_iou_threshold"]).plot(marker="o", title="AP versus IoU threshold")
plt.ylabel("mean AP"); plt.show()

In [ ]:
from IPython.display import Image, display
display(Image(filename=detection["error_figure"]))

## Efficiency frontier

Parameter count is a storage proxy; measured batch latency depends on hardware, threads, and runtime.
Compare both with accuracy rather than assuming smaller always means faster.

In [ ]:
zoo.plot.scatter(x="parameters", y="batch32_inference_ms", s=100, logx=True, logy=True)
for name, row in zoo.iterrows():
    plt.annotate(name, (row.parameters, row.batch32_inference_ms))
plt.title("Model size versus measured latency"); plt.show()

## Exercises and interview checkpoint

1. Run five seeds and plot accuracy confidence intervals.
2. Add brightness augmentation and justify label preservation.
3. Replace the one-object protocol with multiple objects and NMS.
4. Explain why mAP@0.50 is much higher than mAP@0.50:0.95.
5. Identify one defining source-code block for every architecture.